In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("TDM_API_TOKEN")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"TDM_API_TOKEN loaded ({len(token)} characters): {masked}")
else:
    print("TDM_API_TOKEN not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'TDM_API_TOKEN', and load_dotenv() ran without error.")

TDM_API_TOKEN loaded (36 characters): dd9a...5d98


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("ELSEVIER_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"ELSEVIER_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("ELSEVIER_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'ELSEVIER_API_KEY', and load_dotenv() ran without error.")

ELSEVIER_API_KEY loaded (32 characters): cbc1...9f1c


In [5]:
"""
PDF Downloader — Open Access Only
=================================
Downloads freely available open-access PDFs. No institutional proxy,
no eID, no password, no Duo.

Two passes:
  Pass 1 — Unpaywall: free OA PDFs via requests (no browser needed)
  Pass 2 — MDPI via Chrome: MDPI is fully open access but sits behind
           Cloudflare, which blocks plain requests. A real browser gets
           through. Chrome is launched only if MDPI DOIs remain, and it
           visits mdpi.com directly (no proxy, no login).

Requirements:
    pip install unpywall requests selenium webdriver-manager

Usage:
    1. Place DOI lists (.txt files) in doi_files/
    2. Run. Chrome opens only for the MDPI pass; leave it alone until done.
"""

import os
import re
import time
import warnings
import requests
from unpywall import Unpywall
from unpywall.utils import UnpywallCredentials

warnings.filterwarnings("ignore")

# ─── CONFIGURATION ────────────────────────────────────────────────────────────

UNPAYWALL_EMAIL = "olagunju@ksu.edu"

DOI_FOLDER    = "doi_files"
OUTPUT_FOLDER = "PDF_files2"
LOG_SUCCESS   = "OpenAccess_downloads.txt"
LOG_FAILED    = "ClosedAccess_failed.txt"

DOWNLOAD_WAIT = 20   # seconds to wait for a file to appear after navigation
PAGE_LOAD     = 4    # seconds to wait for a page to load

# ─── PASS 1: UNPAYWALL ────────────────────────────────────────────────────────

def try_unpaywall(doi, output_folder):
    """
    Check Unpaywall for a free legal PDF via direct requests.
    Tries /pdf suffix fallback for MDPI-style URLs.
    """
    try:
        UnpywallCredentials(UNPAYWALL_EMAIL)
        pdf_url = Unpywall.get_pdf_link(doi=doi)
        if not pdf_url:
            return False
        for url in [pdf_url, pdf_url.rstrip("/") + "/pdf"]:
            try:
                r = requests.get(url, timeout=30, verify=False, allow_redirects=True)
                if r.status_code == 200 and b"%PDF" in r.content[:10]:
                    safe_name = doi.replace("/", "_").replace(".", "-") + ".pdf"
                    with open(os.path.join(output_folder, safe_name), "wb") as f:
                        f.write(r.content)
                    print(f"  [UNPAYWALL] Downloaded: {doi}")
                    return True
            except:
                continue
    except Exception as e:
        print(f"  [UNPAYWALL] Error: {doi} - {e}")
    return False


# ─── FILE DETECTION ───────────────────────────────────────────────────────────

def wait_for_new_pdf(output_folder, before_set, seconds=20):
    """
    Watch the output folder for a new completed PDF to appear.
    Ignores .crdownload files (Chrome partial downloads).
    Returns the filename if found, None if timeout.
    """
    deadline = time.time() + seconds
    while time.time() < deadline:
        current = set(os.listdir(output_folder))
        new     = current - before_set
        pdfs    = [f for f in new
                   if f.endswith(".pdf") and not f.endswith(".crdownload")]
        if pdfs:
            return pdfs[0]
        time.sleep(1)
    return None


def rename_to_doi(filename, doi, output_folder):
    """Rename Chrome's auto-generated filename to the DOI-based name."""
    old_path  = os.path.join(output_folder, filename)
    safe_name = doi.replace("/", "_").replace(".", "-") + ".pdf"
    new_path  = os.path.join(output_folder, safe_name)
    if old_path != new_path:
        os.rename(old_path, new_path)
    return new_path


# ─── PASS 2: BROWSER (MDPI ONLY) ──────────────────────────────────────────────

def build_driver(download_folder):
    """
    Configure Chrome to auto-save PDFs to the output folder.
    No proxy and no credentials. Used only for open-access MDPI articles.
    """
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager

    abs_folder = os.path.abspath(download_folder)
    prefs = {
        "download.default_directory"         : abs_folder,
        "download.prompt_for_download"       : False,
        "plugins.always_open_pdf_externally"  : True,
        "download.directory_upgrade"         : True,
        "safebrowsing.enabled"               : True,
    }

    options = Options()
    options.add_experimental_option("prefs", prefs)
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.set_page_load_timeout(30)
    return driver


def resolve_doi(doi):
    """Resolve a DOI to the real publisher URL."""
    try:
        r = requests.head(f"https://doi.org/{doi}",
                          allow_redirects=True, timeout=15, verify=False)
        return r.url
    except Exception as e:
        print(f"  [ERROR] Cannot resolve {doi}: {e}")
        return None


def download_mdpi(driver, doi, real_url, output_folder):
    """
    MDPI strategy:
    MDPI is open access. Navigate Chrome (no proxy) to the article page,
    then to the /pdf URL. MDPI blocks requests but not real browsers.
    """
    pdf_url = real_url.rstrip("/") + "/pdf"
    before  = set(os.listdir(output_folder))

    try:
        driver.get(real_url)
        time.sleep(PAGE_LOAD)
    except:
        pass

    try:
        driver.get(pdf_url)
    except:
        pass

    filename = wait_for_new_pdf(output_folder, before, DOWNLOAD_WAIT)
    if filename:
        rename_to_doi(filename, doi, output_folder)
        return True
    return False


def run_mdpi_pass(mdpi_dois, output_folder):
    """
    Launch Chrome once and work through the MDPI DOIs that Unpaywall missed.
    Returns the list of DOIs successfully downloaded.
    """
    recovered = []
    driver = None
    try:
        driver = build_driver(output_folder)
        for doi in mdpi_dois:
            real_url = resolve_doi(doi)
            if not real_url:
                print(f"  [FAILED] {doi}")
                continue
            if "mdpi.com" not in real_url:
                print(f"  [SKIP] {doi} resolved off-site: {real_url}")
                continue
            if download_mdpi(driver, doi, real_url, output_folder):
                print(f"  [OK] Saved: {doi}")
                recovered.append(doi)
            else:
                print(f"  [FAILED] {doi}")
            time.sleep(2)
    except Exception as e:
        print(f"  [MDPI PASS] Browser error: {e}")
    finally:
        if driver:
            driver.quit()
    return recovered


# ─── UTILITIES ────────────────────────────────────────────────────────────────

def extract_doi_from_line(line):
    line = line.strip()
    if not line or line.lower().startswith("doi"):
        return None
    if "\t" in line:
        line = line.split("\t")[0].strip()
    if line.startswith("http"):
        match = re.search(r'(10\.\d{4,}/\S+)', line)
        return match.group(1) if match else None
    if line.startswith("10."):
        return line
    return None


def load_dois(doi_folder):
    all_dois = []
    for fname in sorted(os.listdir(doi_folder)):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(doi_folder, fname), "r") as f:
            dois = [d for d in (extract_doi_from_line(l) for l in f) if d]
        print(f"  Loaded {len(dois)} DOIs from {fname}")
        all_dois.extend(dois)
    return list(dict.fromkeys(all_dois))


def already_downloaded(doi, output_folder):
    safe_name = doi.replace("/", "_").replace(".", "-") + ".pdf"
    return os.path.exists(os.path.join(output_folder, safe_name))


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def run():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(DOI_FOLDER, exist_ok=True)

    dois = load_dois(DOI_FOLDER)
    print(f"\nTotal unique DOIs: {len(dois)}\n")

    success, failed = [], []

    # ── Pass 1: Unpaywall ─────────────────────────────────────────────────────
    print("=== PASS 1: Unpaywall (open access, no browser) ===")
    for doi in dois:
        if already_downloaded(doi, OUTPUT_FOLDER):
            print(f"  [SKIP] Already downloaded: {doi}")
            success.append(doi)
            continue
        if try_unpaywall(doi, OUTPUT_FOLDER):
            success.append(doi)
        else:
            failed.append(doi)

    # ── Pass 2: MDPI via Chrome ───────────────────────────────────────────────
    mdpi_failed = [d for d in failed if d.startswith("10.3390/")]
    if mdpi_failed:
        print(f"\n=== PASS 2: MDPI via Chrome ({len(mdpi_failed)} DOIs) ===")
        print("NOTE: Chrome will open. Leave it alone until the pass finishes.\n")
        recovered = run_mdpi_pass(mdpi_failed, OUTPUT_FOLDER)
        for doi in recovered:
            success.append(doi)
            failed.remove(doi)

    # ── Logs ──────────────────────────────────────────────────────────────────
    with open(LOG_SUCCESS, "w") as f:
        f.write("\n".join(success))
    with open(LOG_FAILED, "w") as f:
        f.write("\n".join(failed))

    print(f"\n{'='*60}")
    print(f"=== FINAL SUMMARY ===")
    print(f"{'='*60}")
    print(f"  Total DOIs processed    : {len(dois)}")
    print(f"  Open-access downloaded  : {len(success)}")
    print(f"  Not open access         : {len(failed)}")
    if dois:
        print(f"  Success rate            : {len(success)/len(dois)*100:.1f}%")

    if failed:
        print(f"\n--- DOIs with no open-access copy found ({len(failed)}) ---")
        for i, doi in enumerate(failed, 1):
            print(f"  {i:3}. {doi}")
        print(f"\n  These have been saved to: {LOG_FAILED}")
        print(f"  Options:")
        print(f"    1. Institutional access : lib.k-state.edu (log in manually)")
        print(f"    2. ILL request          : lib.k-state.edu/interlibrary-loan")
        print(f"    3. Email authors        : search corresponding author on Google Scholar")
        print(f"{'='*60}")


if __name__ == "__main__":
    run()

  Loaded 230 DOIs from Relevant_DOI_250_ARTICLES.txt

Total unique DOIs: 230

=== PASS 1: Unpaywall (open access, no browser) ===
  [UNPAYWALL] Downloaded: 10.1007/s13197-019-03785-8
  [UNPAYWALL] Downloaded: 10.1007/s11694-023-01889-6
  [UNPAYWALL] Downloaded: 10.1038/s41598-026-35526-1
  [UNPAYWALL] Downloaded: 10.3136/fstr.21.95
  [UNPAYWALL] Downloaded: 10.1590/fst.03918
  [UNPAYWALL] Downloaded: 10.1007/s13197-014-1681-3
  [UNPAYWALL] Downloaded: 10.1590/1678-457X.00817
  [UNPAYWALL] Downloaded: 10.1007/s00217-026-05065-0
  [UNPAYWALL] Downloaded: 10.5433/1679-0359.2025v46n3p675
  [UNPAYWALL] Downloaded: 10.3389/fnut.2022.852225
  [UNPAYWALL] Downloaded: 10.1186/s43014-021-00076-8
  [UNPAYWALL] Downloaded: 10.1590/S1516-89132009000600027
  [UNPAYWALL] Downloaded: 10.3136/fstr.11.46
  [UNPAYWALL] Downloaded: 10.3389/fnut.2025.1708593
  [UNPAYWALL] Downloaded: 10.3389/fnut.2021.782212

=== PASS 2: MDPI via Chrome (19 DOIs) ===
NOTE: Chrome will open. Leave it alone until the pass fi

# Download WILEY

In [6]:
"""
Wiley TDM Downloader (official client wrapper)
==============================================
Thin wrapper around WileyLabs' own `wiley-tdm` package. The client does the
API work, pacing, retries and results CSV. This script adds the one thing it
does not do: FILTER YOUR DOI LIST TO WILEY DOIs ONLY.

That filter matters. download_pdfs() attempts every DOI you hand it, so
pointing it straight at download_failed.txt would grind through 114 Elsevier
DOIs at 10 seconds each for guaranteed failures. This script passes only the
10.1111 and 10.1002 DOIs.

HOW ACCESS WORKS (two separate checks):
  1. The token proves you accepted Wiley's TDM license. Get one with your
     ORCID iD at:
       https://onlinelibrary.wiley.com/library-info/resources/text-and-datamining
     Ask K-State Libraries first whether an institutional Wiley TDM agreement
     already exists, since its terms may supersede the click-through license.
  2. Your K-State subscription is checked by IP ADDRESS, not by eID login.
     Run this on the campus network or the K-State VPN. Your eID, password
     and Duo are not used by this API.

SETUP:
    python3 -m venv venv
    source venv/bin/activate          # Windows: venv\\Scripts\\activate
    pip install wiley-tdm python-dotenv

    Create a .env file in this same folder with:
        TDM_API_TOKEN=your-actual-token-here

USAGE:
    python wiley_download.py                    # reads download_failed.txt
    python wiley_download.py my_dois.txt        # or any DOI list
"""

import os
import re
import sys
import shutil
from pathlib import Path

from dotenv import load_dotenv
from wiley_tdm import TDMClient, DownloadStatus

# ─── CONFIGURATION ────────────────────────────────────────────────────────────

INPUT_FILE   = "ClosedAccess_failed.txt"
WILEY_DIR    = Path("wiley_downloads")   # where the client writes
FINAL_DIR    = Path("PDF_files2")         # your main corpus folder
RESULTS_CSV  = "wiley_results.csv"

PREFIXES = ("10.1111/", "10.1002/")

# Wiley's docs recommend 10 seconds for sustained use (60 requests per
# 10 minutes). The package default is 5.0. Do not lower this: aggressive
# downloading gets institutional IP ranges suspended, which would cut off
# Wiley access for everyone at K-State.
RATE_LIMIT = 10.0

# ─── HELPERS ──────────────────────────────────────────────────────────────────

def pick_input_file():
    """
    Return the DOI list to read.

    In a terminal, sys.argv[1] is the filename you passed. In Jupyter,
    sys.argv[1] is the kernel's own argument, which must be ignored.
    """
    for arg in sys.argv[1:]:
        if arg.startswith("-") or arg.endswith(".json"):
            continue
        if os.path.exists(arg):
            return arg
    return INPUT_FILE


def extract_doi(line):
    line = line.strip()
    if not line or line.lower().startswith("doi"):
        return None
    if "\t" in line:
        line = line.split("\t")[0].strip()
    if line.startswith("http"):
        m = re.search(r'(10\.\d{4,}/\S+)', line)
        return m.group(1) if m else None
    return line if line.startswith("10.") else None


def load_dois(path):
    with open(path) as f:
        dois = [d for d in (extract_doi(l) for l in f) if d]
    return list(dict.fromkeys(dois))


def corpus_name(doi):
    """The naming convention used by your other download scripts."""
    return doi.replace("/", "_").replace(".", "-") + ".pdf"


def already_in_corpus(doi):
    return (FINAL_DIR / corpus_name(doi)).exists()


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def run():
    load_dotenv()  # reads TDM_API_TOKEN from .env in this folder

    if not os.environ.get(TDMClient.API_TOKEN_ENV):
        print(f"{TDMClient.API_TOKEN_ENV} is not set.")
        print("Get a token with your ORCID iD at:")
        print("  https://onlinelibrary.wiley.com/library-info/resources/text-and-datamining")
        print(f"\nAdd it to your .env file in this folder:")
        print(f"    {TDMClient.API_TOKEN_ENV}=your-actual-token-here")
        return

    path = pick_input_file()
    if not os.path.exists(path):
        print(f"Input file not found: {path}")
        print(f"Working directory is: {os.getcwd()}")
        return

    FINAL_DIR.mkdir(exist_ok=True)
    WILEY_DIR.mkdir(exist_ok=True)

    all_dois = load_dois(path)
    wiley    = [d for d in all_dois if d.startswith(PREFIXES)]
    todo     = [d for d in wiley if not already_in_corpus(d)]

    print(f"Read {len(all_dois)} DOIs from {path}")
    print(f"Wiley DOIs (10.1111, 10.1002): {len(wiley)}")
    if len(wiley) != len(todo):
        print(f"Already in {FINAL_DIR}/: {len(wiley) - len(todo)}")

    if not todo:
        print("Nothing to do.")
        return

    print(f"To download: {len(todo)}")
    print(f"Estimated time at {RATE_LIMIT}s per request: "
          f"{len(todo) * RATE_LIMIT / 60:.1f} minutes\n")

    try:
        tdm = TDMClient(download_dir=WILEY_DIR)
    except ValueError as e:
        print(f"Could not create client: {e}")
        print("The token must be a valid UUID, e.g. 1234abcd-56ef-...")
        return

    tdm.api_rate_limit = RATE_LIMIT

    def progress(result):
        n = len(tdm.results)
        mark = "OK    " if result.status == DownloadStatus.SUCCESS else result.status.name
        size = f"  ({result.size // 1024} KB)" if result.size else ""
        print(f"[{n}/{len(todo)}] {mark:14} {result.doi}{size}")

    results = tdm.download_pdfs(todo, on_result=progress)
    tdm.save_results(RESULTS_CSV)

    # Move successes into the main corpus folder under the shared naming
    # convention, so your other scripts see them as already downloaded.
    moved = 0
    for r in results:
        if r.status == DownloadStatus.SUCCESS and r.path:
            src = Path(r.path)
            if src.exists():
                shutil.move(str(src), str(FINAL_DIR / corpus_name(r.doi)))
                moved += 1

    # Summarise by status so failures are diagnosable at a glance
    counts = {}
    for r in results:
        counts[r.status.name] = counts.get(r.status.name, 0) + 1

    print(f"\n{'='*60}")
    print(f"  Attempted : {len(results)}")
    for status, n in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"    {status:16} {n}")
    print(f"  Moved into {FINAL_DIR}/ : {moved}")
    print(f"  Full log  : {RESULTS_CSV}")

    if counts.get("ACCESS_DENIED"):
        print("\n  ACCESS_DENIED usually means one of:")
        print("    - you are not on a K-State IP (connect to campus Wi-Fi or VPN)")
        print("    - K-State does not subscribe to that specific journal")
        print("    - the token is not active yet")
        print("  Wiley's check: compare the IP in the client log against the IP")
        print("  your browser shows on Wiley Online Library. If they differ, ask")
        print("  your network admin. If they match, contact tdm@wiley.com.")
    print(f"{'='*60}")


if __name__ == "__main__":
    run()

Read 196 DOIs from ClosedAccess_failed.txt
Wiley DOIs (10.1111, 10.1002): 38
To download: 38
Estimated time at 10.0s per request: 6.3 minutes

[1/38] OK             10.1111/1750-3841.16467  (1 KB)
[2/38] OK             10.1111/j.1750-3841.2006.00156.x  (0 KB)
[3/38] OK             10.1111/jfpp.12507  (0 KB)
[4/38] OK             10.1002/fsn3.4511  (1 KB)
[5/38] OK             10.1111/jfpp.12488  (0 KB)
[6/38] OK             10.1111/ijfs.16923  (1 KB)
[7/38] OK             10.1002/jsfa.12702  (1 KB)
[8/38] OK             10.1111/jfpe.70538  (1 KB)
[9/38] OK             10.1002/leg3.70032  (0 KB)
[10/38] OK             10.1002/cche.10383  (0 KB)
[11/38] OK             10.1002/leg3.70005  (0 KB)
[12/38] OK             10.1002/food.200390094  (0 KB)


DOI: 10.1111/j.1365-2621.1983.tb14882.x - Access Denied


[13/38] ACCESS_DENIED  10.1111/j.1365-2621.1983.tb14882.x
[14/38] OK             10.1111/ijfs.13458  (0 KB)
[15/38] OK             10.1111/ijfs.15446  (4 KB)
[16/38] OK             10.1111/j.1365-2621.2002.tb10299.x  (0 KB)
[17/38] OK             10.1111/ijfs.13271  (0 KB)


DOI: 10.1111/j.1365-2621.1986.tb13096.x - Access Denied


[18/38] ACCESS_DENIED  10.1111/j.1365-2621.1986.tb13096.x
[19/38] OK             10.1111/1750-3841.16794  (1 KB)
[20/38] OK             10.1002/jsfa.70829  (2 KB)
[21/38] OK             10.1111/ijfs.14969  (0 KB)
[22/38] OK             10.1111/ijfs.17244  (0 KB)
[23/38] OK             10.1111/ijfs.13990  (1 KB)
[24/38] OK             10.1002/jsfa.2334  (0 KB)
[25/38] OK             10.1002/cche.10589  (1 KB)
[26/38] OK             10.1111/jfpe.14243  (2 KB)
[27/38] OK             10.1111/jfpe.14578  (2 KB)
[28/38] OK             10.1111/ijfs.17596  (1 KB)
[29/38] OK             10.1111/ijfs.15831  (1 KB)
[30/38] OK             10.1111/1750-3841.14770  (1 KB)
[31/38] OK             10.1111/ijfs.17242  (0 KB)
[32/38] OK             10.1111/jfpp.16764  (1 KB)
[33/38] OK             10.1111/jfpp.13524  (0 KB)
[34/38] OK             10.1111/j.1365-2621.2005.tb09023.x  (0 KB)


Error validating DOI 10.1111/1750-3841.17607: HTTPSConnectionPool(host='doi.org', port=443): Read timed out. (read timeout=3)


[35/38] OK             10.1111/1750-3841.17607  (1 KB)
[36/38] OK             10.1002/jsfa.13233  (0 KB)


DOI: 10.1002/(SICI)1521-3803(199808)42:03/04<245::AID-FOOD245>3.3.CO;2-Q - Unknown Doi


[37/38] UNKNOWN_DOI    10.1002/(SICI)1521-3803(199808)42:03/04<245::AID-FOOD245>3.3.CO;2-Q
[38/38] OK             10.1111/jfpp.16218  (0 KB)

  Attempted : 38
    SUCCESS          35
    ACCESS_DENIED    2
    UNKNOWN_DOI      1
  Moved into PDF_files2/ : 35
  Full log  : wiley_results.csv

  ACCESS_DENIED usually means one of:
    - you are not on a K-State IP (connect to campus Wi-Fi or VPN)
    - K-State does not subscribe to that specific journal
    - the token is not active yet
  Wiley's check: compare the IP in the client log against the IP
  your browser shows on Wiley Online Library. If they differ, ask
  your network admin. If they match, contact tdm@wiley.com.


In [1]:
import os
import re
import sys
import shutil
from pathlib import Path

from dotenv import load_dotenv
from wiley_tdm import TDMClient, DownloadStatus

# ─── CONFIGURATION ────────────────────────────────────────────────────────────

INPUT_FILE   = "ClosedAccess_failed.txt"
WILEY_DIR    = Path("wiley_downloads")   # where the client writes
FINAL_DIR    = Path("PDF_files2")        # your main corpus folder
REMAINING_OUT = "remaining_after_wiley.txt"

PREFIXES = ("10.1111/", "10.1002/")

# Wiley's docs recommend 10 seconds for sustained use (60 requests per
# 10 minutes). The package default is 5.0. Do not lower this: aggressive
# downloading gets institutional IP ranges suspended, which would cut off
# Wiley access for everyone at K-State.
RATE_LIMIT = 10.0

# ─── HELPERS ──────────────────────────────────────────────────────────────────

def pick_input_file():
    """
    Return the DOI list to read.

    In a terminal, sys.argv[1] is the filename you passed. In Jupyter,
    sys.argv[1] is the kernel's own argument, which must be ignored.
    """
    for arg in sys.argv[1:]:
        if arg.startswith("-") or arg.endswith(".json"):
            continue
        if os.path.exists(arg):
            return arg
    return INPUT_FILE


def extract_doi(line):
    line = line.strip()
    if not line or line.lower().startswith("doi"):
        return None
    if "\t" in line:
        line = line.split("\t")[0].strip()
    if line.startswith("http"):
        m = re.search(r'(10\.\d{4,}/\S+)', line)
        return m.group(1) if m else None
    return line if line.startswith("10.") else None


def load_dois(path):
    with open(path) as f:
        dois = [d for d in (extract_doi(l) for l in f) if d]
    return list(dict.fromkeys(dois))


def corpus_name(doi):
    """The naming convention used by your other download scripts."""
    return doi.replace("/", "_").replace(".", "-") + ".pdf"


def already_in_corpus(doi):
    return (FINAL_DIR / corpus_name(doi)).exists()


def report_remaining(all_dois):
    """
    Write and print every DOI in the input that is still not in FINAL_DIR,
    Elsevier included. One file, so it can be fed straight to the next
    script in the chain.
    Runs whether or not the Wiley pass had anything to do.
    """
    remaining = [d for d in all_dois if not already_in_corpus(d)]

    with open(REMAINING_OUT, "w") as f:
        f.write("\n".join(remaining))

    print(f"\n  Still undownloaded: {len(remaining)} of {len(all_dois)}"
          f"   -> {REMAINING_OUT}")

    # Breakdown by prefix, so you can see what the next script is facing
    groups = {}
    for doi in remaining:
        prefix = doi.split("/")[0]
        groups[prefix] = groups.get(prefix, 0) + 1
    for prefix, n in sorted(groups.items(), key=lambda x: -x[1]):
        print(f"    {prefix:12} {n}")

    for i, doi in enumerate(remaining, 1):
        print(f"    {i:3}. {doi}")

# ─── MAIN ─────────────────────────────────────────────────────────────────────

def run():
    load_dotenv()  # reads TDM_API_TOKEN from .env in this folder

    if not os.environ.get(TDMClient.API_TOKEN_ENV):
        print(f"{TDMClient.API_TOKEN_ENV} is not set.")
        print("Get a token with your ORCID iD at:")
        print("  https://onlinelibrary.wiley.com/library-info/resources/text-and-datamining")
        print(f"\nAdd it to your .env file in this folder:")
        print(f"    {TDMClient.API_TOKEN_ENV}=your-actual-token-here")
        return

    path = pick_input_file()
    if not os.path.exists(path):
        print(f"Input file not found: {path}")
        print(f"Working directory is: {os.getcwd()}")
        return

    FINAL_DIR.mkdir(exist_ok=True)
    WILEY_DIR.mkdir(exist_ok=True)

    all_dois = load_dois(path)
    wiley    = [d for d in all_dois if d.startswith(PREFIXES)]
    todo     = [d for d in wiley if not already_in_corpus(d)]

    print(f"Read {len(all_dois)} DOIs from {path}")
    print(f"Wiley DOIs (10.1111, 10.1002): {len(wiley)}")
    if len(wiley) != len(todo):
        print(f"Already in {FINAL_DIR}/: {len(wiley) - len(todo)}")

    if not todo:
        print("No Wiley DOIs left to download.")
        report_remaining(all_dois)
        return

    print(f"To download: {len(todo)}")
    print(f"Estimated time at {RATE_LIMIT}s per request: "
          f"{len(todo) * RATE_LIMIT / 60:.1f} minutes\n")

    try:
        tdm = TDMClient(download_dir=WILEY_DIR)
    except ValueError as e:
        print(f"Could not create client: {e}")
        print("The token must be a valid UUID, e.g. 1234abcd-56ef-...")
        return

    tdm.api_rate_limit = RATE_LIMIT

    def progress(result):
        n = len(tdm.results)
        mark = "OK    " if result.status == DownloadStatus.SUCCESS else result.status.name
        size = f"  ({result.size // 1024} KB)" if result.size else ""
        print(f"[{n}/{len(todo)}] {mark:14} {result.doi}{size}")

    results = tdm.download_pdfs(todo, on_result=progress)

    # Move successes into the main corpus folder under the shared naming
    # convention, so your other scripts see them as already downloaded.
    moved = 0
    for r in results:
        if r.status == DownloadStatus.SUCCESS and r.path:
            src = Path(r.path)
            if src.exists():
                shutil.move(str(src), str(FINAL_DIR / corpus_name(r.doi)))
                moved += 1

    # Summarise by status so failures are diagnosable at a glance
    counts = {}
    for r in results:
        counts[r.status.name] = counts.get(r.status.name, 0) + 1

    print(f"\n{'='*60}")
    print(f"  Attempted : {len(results)}")
    for status, n in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"    {status:16} {n}")
    print(f"  Moved into {FINAL_DIR}/ : {moved}")

    report_remaining(all_dois)

    if counts.get("ACCESS_DENIED"):
        print("\n  ACCESS_DENIED usually means one of:")
        print("    - you are not on a K-State IP (connect to campus Wi-Fi or VPN)")
        print("    - K-State does not subscribe to that specific journal")
        print("    - the token is not active yet")
        print("  Wiley's check: compare the IP in the client log against the IP")
        print("  your browser shows on Wiley Online Library. If they differ, ask")
        print("  your network admin. If they match, contact tdm@wiley.com.")
    print(f"{'='*60}")


if __name__ == "__main__":
    run()

Read 196 DOIs from ClosedAccess_failed.txt
Wiley DOIs (10.1111, 10.1002): 38
Already in PDF_files2/: 35
To download: 3
Estimated time at 10.0s per request: 0.5 minutes



DOI: 10.1111/j.1365-2621.1983.tb14882.x - Access Denied


[1/3] ACCESS_DENIED  10.1111/j.1365-2621.1983.tb14882.x


DOI: 10.1111/j.1365-2621.1986.tb13096.x - Access Denied


[2/3] ACCESS_DENIED  10.1111/j.1365-2621.1986.tb13096.x


DOI: 10.1002/(SICI)1521-3803(199808)42:03/04<245::AID-FOOD245>3.3.CO;2-Q - Unknown Doi


[3/3] UNKNOWN_DOI    10.1002/(SICI)1521-3803(199808)42:03/04<245::AID-FOOD245>3.3.CO;2-Q

  Attempted : 3
    ACCESS_DENIED    2
    UNKNOWN_DOI      1
  Moved into PDF_files2/ : 0

  Still undownloaded: 161 of 196   -> remaining_after_wiley.txt
    10.1016      114
    10.1007      18
    10.1021      7
    10.1080      4
    10.1556      3
    10.1177      3
    10.21603     2
    10.1515      2
    10.1111      2
    10.14233     1
    10.3109      1
    10.3920      1
    10.37527     1
    10.1155      1
    10.1002      1
      1. 10.1080/00986445.2012.717316
      2. 10.1016/j.foodchem.2006.01.011
      3. 10.1016/j.ijbiomac.2025.143552
      4. 10.14233/ajchem.2014.15595
      5. 10.21603/2308-4057-2023-2-566
      6. 10.1016/j.foodchem.2021.131809
      7. 10.1515/ijfe-2018-0388
      8. 10.1016/j.foodhyd.2022.108085
      9. 10.1016/j.ijbiomac.2017.11.135
     10. 10.1016/j.jafr.2022.100313
     11. 10.1016/j.lwt.2023.114585
     12. 10.1016/j.fbio.2023.102761
     13. 10.101

In [2]:
from collections import Counter

FILE = "remaining_after_wiley.txt"

dois = [l.strip() for l in open(FILE) if l.strip()]

print(f"{FILE}: {len(dois)} DOIs  ({len(set(dois))} unique)\n")

for prefix, n in Counter(d.split("/")[0] for d in dois).most_common():
    print(f"  {prefix:12} {n}")

remaining_after_wiley.txt: 161 DOIs  (161 unique)

  10.1016      114
  10.1007      18
  10.1021      7
  10.1080      4
  10.1556      3
  10.1177      3
  10.21603     2
  10.1515      2
  10.1111      2
  10.14233     1
  10.3109      1
  10.3920      1
  10.37527     1
  10.1155      1
  10.1002      1


# Download Elsevier

In [3]:
"""
Elsevier Full-Text API Fetcher
===============================
Reads download_failed.txt, fetches full text for all Elsevier DOIs
via the Elsevier TDM (Text and Data Mining) API.

Requirements:
    - Must be on K-State network or K-State VPN for full-text access
    - pip install requests python-dotenv

Setup:
    Create a .env file in this same folder with:
        ELSEVIER_API_KEY=your-actual-key-here

Output:
    - Full text saved as .txt files in TXT_OUTPUT folder
    - These feed directly into your preprocessing notebook
      (skips Azure OCR entirely for Elsevier papers)
    - download_failed_remaining.txt lists any DOIs still not retrieved

Usage:
    1. Make sure .env contains your Elsevier API key
    2. Make sure download_failed.txt exists (produced by pdf_downloader.py)
    3. Run: python elsevier_api_fetch.py
"""

import os
import time
import requests
from dotenv import load_dotenv

# ── CONFIGURATION ─────────────────────────────────────────────────────────────

FAILED_LOG     = "remaining_after_wiley.txt" # written by pdf_downloader.py
TXT_OUTPUT     = "Elsevier_TXT_files3"           # folder for retrieved full texts
STILL_FAILED   = "download_failed_remaining.txt"

# ── HELPERS ───────────────────────────────────────────────────────────────────

def load_failed_dois(filepath):
    """Read all DOIs from download_failed.txt."""
    if not os.path.exists(filepath):
        print(f"[ERROR] {filepath} not found.")
        print("        Run pdf_downloader.py first to generate this file.")
        return []
    with open(filepath, "r") as f:
        dois = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(dois)} failed DOIs from {filepath}")
    return dois


def is_elsevier(doi):
    """
    Elsevier DOIs start with 10.1016 (ScienceDirect journals).
    Other Elsevier prefixes are included below as well.
    """
    elsevier_prefixes = [
        "10.1016",   # ScienceDirect (main)
        "10.1006",   # older Academic Press journals
        "10.1053",   # some clinical journals
        "10.1054",
        "10.1067",
    ]
    return any(doi.startswith(p) for p in elsevier_prefixes)


def already_fetched(doi, output_folder):
    safe_name = doi.replace("/", "_").replace(".", "-") + ".txt"
    return os.path.exists(os.path.join(output_folder, safe_name))


def fetch_elsevier_fulltext(doi, api_key, output_folder):
    """
    Fetch full text for one Elsevier DOI via the TDM API.

    Returns:
        "full"     -- full article text retrieved and saved
        "abstract" -- only abstract returned (not on K-State network)
        "failed"   -- API error or no content
    """
    url     = f"https://api.elsevier.com/content/article/doi/{doi}"
    headers = {
        "X-ELS-APIKey" : api_key,
        "Accept"       : "text/plain",   # clean UTF-8 text, no XML markup
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)

        if response.status_code == 200:
            text      = response.text
            word_count = len(text.split())

            # A full article is typically 3000+ words
            # An abstract-only response is usually under 500 words
            if word_count < 500:
                print(f"  [ABSTRACT ONLY] {doi} ({word_count} words)")
                print(f"  You may not be on K-State network/VPN.")
                return "abstract"

            # Save full text
            safe_name = doi.replace("/", "_").replace(".", "-") + ".txt"
            out_path  = os.path.join(output_folder, safe_name)
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(text)
            print(f"  [OK] {doi} ({word_count} words)")
            return "full"

        elif response.status_code == 401:
            print(f"  [AUTH ERROR] Invalid API key for: {doi}")
            return "failed"

        elif response.status_code == 403:
            print(f"  [ACCESS DENIED] No entitlement for: {doi}")
            print(f"  Check K-State network connection or VPN.")
            return "failed"

        elif response.status_code == 404:
            print(f"  [NOT FOUND] DOI not in Elsevier API: {doi}")
            return "failed"

        else:
            print(f"  [ERROR] Status {response.status_code} for: {doi}")
            print(f"  Response: {response.text[:200]}")
            return "failed"

    except Exception as e:
        print(f"  [EXCEPTION] {doi}: {e}")
        return "failed"


# ── MAIN ──────────────────────────────────────────────────────────────────────

def run():
    load_dotenv()  # reads ELSEVIER_API_KEY from .env in this folder

    api_key = os.environ.get("ELSEVIER_API_KEY")
    if not api_key:
        print("ELSEVIER_API_KEY is not set.")
        print("Add it to your .env file in this folder:")
        print("    ELSEVIER_API_KEY=your-actual-key-here")
        return

    os.makedirs(TXT_OUTPUT, exist_ok=True)

    all_failed   = load_failed_dois(FAILED_LOG)
    elsevier     = [d for d in all_failed if is_elsevier(d)]
    non_elsevier = [d for d in all_failed if not is_elsevier(d)]

    print(f"\nElsevier DOIs to fetch : {len(elsevier)}")
    print(f"Non-Elsevier (skipped) : {len(non_elsevier)}")
    print(f"Output folder          : {TXT_OUTPUT}/\n")

    if not elsevier:
        print("No Elsevier DOIs found in failed list.")
        return

    # Test the API key with the first DOI before running all
    print("=== Testing API key with first DOI ===")
    test_result = fetch_elsevier_fulltext(elsevier[0], api_key, TXT_OUTPUT)
    if test_result == "failed":
        print("\n[STOP] API key test failed. Check your key and try again.")
        return
    if test_result == "abstract":
        print("\n[WARNING] Only getting abstracts. Connect to K-State VPN and rerun.")
        return

    print("\n=== Fetching all Elsevier DOIs ===")
    results = {"full": [], "abstract": [], "failed": []}

    for doi in elsevier:
        if already_fetched(doi, TXT_OUTPUT):
            print(f"  [SKIP] Already fetched: {doi}")
            results["full"].append(doi)
            continue
        result = fetch_elsevier_fulltext(doi, api_key, TXT_OUTPUT)
        results[result].append(doi)
        time.sleep(0.5)   # 2 requests/second -- well within API rate limits

    # Write remaining failures
    still_failed = results["abstract"] + results["failed"] + non_elsevier
    with open(STILL_FAILED, "w") as f:
        f.write("\n".join(still_failed))

    print(f"\n{'='*60}")
    print(f"=== ELSEVIER API SUMMARY ===")
    print(f"{'='*60}")
    print(f"  Full text retrieved : {len(results['full'])}")
    print(f"  Abstract only       : {len(results['abstract'])}")
    print(f"  Failed              : {len(results['failed'])}")
    print(f"\n  Non-Elsevier DOIs still needing manual action ({len(non_elsevier)}):")
    for doi in non_elsevier:
        print(f"    {doi}")
    print(f"\n  Remaining failures saved to: {STILL_FAILED}")
    print(f"  Options: ILL at lib.k-state.edu or email corresponding author")
    print(f"{'='*60}")
    print(f"\nNOTE: TXT files in {TXT_OUTPUT}/ feed directly into your")
    print(f"preprocessing notebook, bypassing Azure OCR entirely.")


if __name__ == "__main__":
    run()

Loaded 161 failed DOIs from remaining_after_wiley.txt

Elsevier DOIs to fetch : 114
Non-Elsevier (skipped) : 47
Output folder          : Elsevier_TXT_files3/

=== Testing API key with first DOI ===
  [OK] 10.1016/j.foodchem.2006.01.011 (6762 words)

=== Fetching all Elsevier DOIs ===
  [SKIP] Already fetched: 10.1016/j.foodchem.2006.01.011
  [SKIP] Already fetched: 10.1016/j.ijbiomac.2025.143552
  [SKIP] Already fetched: 10.1016/j.foodchem.2021.131809
  [SKIP] Already fetched: 10.1016/j.foodhyd.2022.108085
  [SKIP] Already fetched: 10.1016/j.ijbiomac.2017.11.135
  [SKIP] Already fetched: 10.1016/j.jafr.2022.100313
  [SKIP] Already fetched: 10.1016/j.lwt.2023.114585
  [SKIP] Already fetched: 10.1016/j.fbio.2023.102761
  [SKIP] Already fetched: 10.1016/S0308-8146(02)00555-1
  [SKIP] Already fetched: 10.1016/j.foodres.2023.113912
  [SKIP] Already fetched: 10.1016/j.ultsonch.2017.01.042
  [SKIP] Already fetched: 10.1016/j.lwt.2013.10.022
  [SKIP] Already fetched: 10.1016/j.foodres.2024.115